[//]: # (cr:doc name='section' id=5dfd2557)


[//]: # (cr:doc name='chapter_5_relationship_analysis' id=f85cadd3)
# Chapter 5: Relationship Analysis

**Purpose:** Explore feature correlations, relationships with the target, and identify predictive signals.

**What you'll learn:**
- How to interpret correlation matrices and identify multicollinearity
- How to visualize feature distributions by target class
- How to identify which features have the strongest relationship with retention
- How to analyze categorical features for predictive power

**Outputs:**
- Correlation heatmap with multicollinearity detection
- Feature distributions by retention status (box plots)
- Retention rates by categorical features
- Feature-target correlation rankings

---

## Understanding Feature Relationships

| Analysis | What It Tells You | Action |
|----------|------------------|--------|
| **High Correlation** (r > 0.7) | Features carry redundant information | Consider removing one |
| **Target Correlation** | Feature's predictive power | Prioritize high-correlation features |
| **Class Separation** | How different retained vs churned look | Good separation = good predictor |
| **Categorical Rates** | Retention varies by category | Use for segmentation and encoding |

[//]: # (cr:doc name='5_1_setup' id=7576bf8b)
## 5.1 Setup

In [ ]:
# @cr:code name='init_progress' id=6850ee7b
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous

accept_workflow_params()
track_and_export_previous("05_relationship_analysis.ipynb")

import numpy as np
import plotly.graph_objects as go
import yaml
from plotly.subplots import make_subplots

from customer_retention.analysis.auto_explorer import ExplorationFindings, ExplorationManager, RecommendationRegistry
from customer_retention.analysis.visualization import ChartBuilder, display_figure, display_table
from customer_retention.core.compat import (
    _is_spark_pandas,
    batched_corr_matrix,
    bulk_corr_with_target,
    bulk_effect_sizes,
    bulk_null_counts,
    head_as_list,
    native_pd,
    safe_len,
    safe_sample,
    spark_checkpoint,
    track_stage_object,
)
from customer_retention.core.config.column_config import ColumnType
from customer_retention.core.config.experiments import FINDINGS_DIR  # noqa: F401
from customer_retention.core.utils.leakage import detect_leaking_features
from customer_retention.stages.profiling import RecommendationCategory, RelationshipRecommender

# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


In [ ]:
# @cr:code name='load_findings' id=09b14f1e
from customer_retention.analysis.auto_explorer import load_notebook_findings

FINDINGS_PATH, _namespace, dataset_name = load_notebook_findings(
    "05_relationship_analysis.ipynb", prefer_merged=True
)
print(f"Using: {FINDINGS_PATH}")

RECOMMENDATIONS_PATH = str(_namespace.merged_recommendations_path)

findings = ExplorationFindings.load(FINDINGS_PATH)

from customer_retention.analysis.auto_explorer.active_dataset_store import (
    require_silver_merged,
    require_silver_merged_distributed,
)
from customer_retention.analysis.auto_explorer.findings import classify_columns
from customer_retention.core.compat import is_databricks
from customer_retention.stages.temporal import TEMPORAL_METADATA_COLS

df = require_silver_merged_distributed(_namespace) if is_databricks() else require_silver_merged(_namespace)
track_stage_object(df)
data_source = "silver_merged"

_df_cols = set(df.columns)
findings.columns = {k: v for k, v in findings.columns.items() if k in _df_cols}
if findings.target_column and findings.target_column not in _df_cols:
    findings.target_column = None

charts = ChartBuilder()
cc = classify_columns(findings, exclude=set(TEMPORAL_METADATA_COLS))

if _namespace.merged_recommendations_path.exists():
    with open(str(_namespace.merged_recommendations_path), "r") as f:
        registry = RecommendationRegistry.from_dict(yaml.safe_load(f))
    print(f"Loaded existing recommendations: {len(registry.all_recommendations)} total")
else:
    registry = RecommendationRegistry()
    registry.init_bronze(findings.source_path)
    _entity_col = (findings.time_series_metadata.entity_column
                   if findings.time_series_metadata else None)
    registry.init_silver(_entity_col or "entity_id")
    registry.init_gold(findings.target_column or "target")
    print("Initialized new recommendation registry")

[//]: # (cr:doc name='5_1b_leakage_exclusion_gate' id=52692d66)
## 5.1b Leakage Exclusion Gate

Features that encode the target outcome (e.g. a "cancelled" flag that is set only after churn) will inflate correlation metrics and produce misleadingly strong recommendations. This gate auto-detects and removes them before any analysis runs, so that downstream correlation, effect-size, and feature-selection results reflect genuinely predictive signals. Add column names to `EXCLUDE_LEAKING_FEATURES` to manually exclude additional columns you suspect of leakage.

In [ ]:
# @cr:code name='cache_analysis_df' id=d2b9dcce
if _is_spark_pandas(df):
    df = spark_checkpoint(df)
    track_stage_object(df)
    print(f"Checkpointed analysis DataFrame: {df.shape}")

In [ ]:
# @cr:code name='precompute_stats' id=a7c3e91f
_check_cols = [
    name for name, col in findings.columns.items()
    if col.inferred_type in [ColumnType.NUMERIC_CONTINUOUS, ColumnType.NUMERIC_DISCRETE, ColumnType.BINARY]
    and name != findings.target_column
    and name not in TEMPORAL_METADATA_COLS
]
_target_corrs = bulk_corr_with_target(df, _check_cols, findings.target_column, progress_fn=print)
_null_counts = bulk_null_counts(df, _check_cols, progress_fn=print)
_row_count = safe_len(df)
_non_null_counts = {c: _row_count - v for c, v in _null_counts.items()}
print(f"Precomputed target correlations for {len(_target_corrs)} columns")
print(f"Precomputed null counts for {len(_null_counts)} columns ({_row_count} rows)")

In [ ]:
# @cr:code name='check_leaking_features' id=ecad955f
EXCLUDE_LEAKING_FEATURES = []

_auto_leakers = detect_leaking_features(
    df, _check_cols, findings.target_column, progress_fn=print,
    precomputed_value_corrs=_target_corrs, precomputed_null_counts=_null_counts,
)
_all_excluded = sorted(set(_auto_leakers) | set(EXCLUDE_LEAKING_FEATURES))

if _all_excluded:
    for _col in _all_excluded:
        findings.columns.pop(_col, None)
    df = df.drop(columns=[c for c in _all_excluded if c in df.columns])
    findings.excluded_leaking_features = _all_excluded

    _auto_only = [c for c in _auto_leakers if c not in EXCLUDE_LEAKING_FEATURES]
    _manual_only = [c for c in EXCLUDE_LEAKING_FEATURES if c not in _auto_leakers]
    print(f"Excluded {len(_all_excluded)} leaking feature(s):")
    if _auto_only:
        print(f"  Auto-detected: {', '.join(_auto_only)}")
    if _manual_only:
        print(f"  Manual: {', '.join(_manual_only)}")
    _both = [c for c in _all_excluded if c in _auto_leakers and c in EXCLUDE_LEAKING_FEATURES]
    if _both:
        print(f"  Both: {', '.join(_both)}")
else:
    print("No leaking features detected.")

# Recompute column classification after leaking feature removal
cc = classify_columns(findings, exclude=set(TEMPORAL_METADATA_COLS))


[//]: # (cr:doc name='5_2_numeric_correlation_matrix' id=245cd8a1)
## 5.2 Numeric Correlation Matrix

Pairwise Pearson correlations reveal linear redundancy among numeric features. Highly correlated pairs (|r| > 0.7) carry largely the same information — keeping both increases model complexity without adding predictive power and destabilises coefficient estimates in linear models. The heatmap below surfaces these pairs so the feature-selection step at the end of this notebook can recommend which to drop.

- **Red (+1)** / **Blue (-1)**: strong positive / negative linear dependency
- **White (0)**: no linear relationship

In [ ]:
# @cr:config name='relationship_config' id=a3923181
MAX_CORR_FEATURES = 60
MAX_TARGET_CORR_BARS = 30
MAX_BOX_FEATURES = 6
MAX_CATEGORICAL_DETAILS = 5
MAX_SCATTER_FEATURES = 4
MAX_DATETIME_DETAILS = 3
SKIP_FULL_CORR_MATRIX = False


In [ ]:
# @cr:code name='compute_correlations' id=1fa22c14
numeric_cols = cc.numeric + ([cc.target] if cc.target else [])

corr_matrix = None
if len(numeric_cols) >= 2 and not SKIP_FULL_CORR_MATRIX:
    if cc.target:
        feature_cols = [c for c in numeric_cols if c != cc.target]
        _target_corrs = {c: v for c, v in _target_corrs.items() if c in feature_cols}

    corr_matrix = batched_corr_matrix(df, numeric_cols, progress_fn=print, precomputed_non_null=_non_null_counts)

    _corr_index = set(corr_matrix.index)
    display_cols = [c for c in numeric_cols if c in _corr_index]
    if len(numeric_cols) > MAX_CORR_FEATURES and cc.target and _target_corrs:
        top_features = sorted(
            (c for c in _target_corrs if c in _corr_index),
            key=lambda c: abs(_target_corrs[c]),
            reverse=True,
        )[:MAX_CORR_FEATURES - 1]
        display_cols = top_features + ([cc.target] if cc.target in _corr_index else [])
        print(f"  Displaying top {len(display_cols)} of {len(numeric_cols)} numeric columns")

    if display_cols:
        display_matrix = corr_matrix.loc[display_cols, display_cols]
        fig = charts.heatmap(
            display_matrix.to_numpy(),
            x_labels=display_matrix.columns.tolist(),
            y_labels=display_matrix.index.tolist(),
            title="Numeric Correlation Matrix"
        )
        display_figure(fig)
    else:
        print("No columns available for correlation display.")
elif SKIP_FULL_CORR_MATRIX:
    print("Full correlation matrix skipped (SKIP_FULL_CORR_MATRIX=True).")
else:
    print("Not enough numeric columns for correlation analysis.")

[//]: # (cr:doc name='5_3_high_correlation_pairs' id=a08cb7b9)
## 5.3 High Correlation Pairs

Extracts all feature pairs exceeding the multicollinearity threshold (|r| >= 0.7). For each pair the weaker predictor — measured by absolute correlation with the target — becomes a candidate for `drop_multicollinear` in the recommendations. This drives the NB05 statistical drops that NB08 applies before training.

In [ ]:
# @cr:code name='find_high_correlations' id=03e56f1f
high_corr_threshold = 0.7
high_corr_pairs = []

if corr_matrix is not None:
    matrix_cols = corr_matrix.columns.tolist()
    for i in range(len(matrix_cols)):
        for j in range(i+1, len(matrix_cols)):
            corr_val = corr_matrix.iloc[i, j]
            if abs(corr_val) >= high_corr_threshold:
                high_corr_pairs.append({
                    "Column 1": matrix_cols[i],
                    "Column 2": matrix_cols[j],
                    "Correlation": f"{corr_val:.3f}"
                })

if high_corr_pairs:
    print(f"High Correlation Pairs (|r| >= {high_corr_threshold}):")
    display_table(native_pd.DataFrame(high_corr_pairs))
    print("\nConsider removing one of each pair to reduce multicollinearity.")
else:
    print("No high correlation pairs detected.")


[//]: # (cr:doc name='5_4_feature_distributions_by_retention_status' id=817a8b34)
## 5.4 Feature Distributions by Retention Status

Compares the distribution of each numeric feature between retained and churned groups using box plots and Cohen's d effect size. Features with large separation (|d| >= 0.8) are strong standalone discriminators; those with negligible separation (|d| < 0.2) contribute little on their own and are candidates for the `drop_weak` recommendation.

- **Clear box separation** with different medians = strong predictor
- **Overlapping boxes** = feature cannot distinguish classes alone

In [ ]:
# @cr:code name='plot_feature_distributions' id=ae2926c9
# Feature Distributions by Retention Status
_effect_sizes_result = None

if findings.target_column and findings.target_column in df.columns:
    target = findings.target_column

    feature_cols = [
        name for name, col in findings.columns.items()
        if col.inferred_type in [ColumnType.NUMERIC_CONTINUOUS, ColumnType.NUMERIC_DISCRETE]
        and name != target
        and name not in TEMPORAL_METADATA_COLS
    ]

    if feature_cols:
        print("=" * 80)
        print(f"FEATURE DISTRIBUTIONS BY TARGET: {target}")
        print("=" * 80)

        # Bulk compute effect sizes and per-class stats (2 Spark agg calls instead of 16N)
        _effect_sizes_result = bulk_effect_sizes(df, feature_cols, target)

        # Build summary table from bulk class_stats
        summary_by_target = []
        for col in feature_cols:
            stats = _effect_sizes_result.class_stats.get(col)
            if not stats:
                continue
            for cls, label in [(0, "Churned"), (1, "Retained")]:
                if stats[f"count_{cls}"] > 0:
                    summary_by_target.append({
                        "Feature": col,
                        "Group": label,
                        "Count": stats[f"count_{cls}"],
                        "Mean": stats[f"mean_{cls}"],
                        "Median": stats[f"median_{cls}"],
                        "Std": stats[f"std_{cls}"],
                    })

        if summary_by_target:
            summary_df = native_pd.DataFrame(summary_by_target)

            # Display summary table
            print("\n📊 Summary Statistics by Retention Status:")
            display_summary = summary_df.pivot(index="Feature", columns="Group", values=["Mean", "Median"])
            display_summary.columns = [f"{stat} ({group})" for stat, group in display_summary.columns]
            display_table(display_summary.round(3))

        # Display effect sizes from bulk result
        print("\n📈 Feature Importance Indicators (Effect Size - Cohen's d):")
        print("-" * 70)
        effect_sizes = []
        for col in feature_cols:
            d = _effect_sizes_result.effect_sizes.get(col)
            if d is None:
                continue

            abs_d = abs(d)
            if abs_d >= 0.8:
                interpretation = "Large effect"
                emoji = "🔴"
            elif abs_d >= 0.5:
                interpretation = "Medium effect"
                emoji = "🟡"
            elif abs_d >= 0.2:
                interpretation = "Small effect"
                emoji = "🟢"
            else:
                interpretation = "Negligible"
                emoji = "⚪"

            effect_sizes.append({
                "feature": col,
                "cohens_d": d,
                "abs_d": abs_d,
                "interpretation": interpretation
            })

            direction = "↑ Higher in retained" if d > 0 else "↓ Lower in retained"
            print(f"  {emoji} {col}: d={d:+.3f} ({interpretation}) {direction}")

        # Sort by effect size for identifying important features
        if effect_sizes:
            effect_df = native_pd.DataFrame(effect_sizes).sort_values("abs_d", ascending=False)
            important_features = effect_df[effect_df["abs_d"] >= 0.2]["feature"].tolist()
            if important_features:
                print(f"\n⭐ Features with notable effect (|d| ≥ 0.2): {', '.join(important_features)}")
        else:
            print("  No effect sizes could be calculated (insufficient data in one or both groups)")
    else:
        print("No numeric feature columns found for distribution analysis.")
else:
    print("Target column not available.")

[//]: # (cr:doc name='interpreting_effect_sizes_cohen_s_d' id=0157c38f)
### Interpreting Effect Sizes (Cohen's d)

| Effect Size | Interpretation | What It Means for Modeling |
|-------------|----------------|---------------------------|
| \|d\| ≥ 0.8 | Large | Strong discriminator - prioritize this feature |
| \|d\| = 0.5-0.8 | Medium | Useful predictor - include in model |
| \|d\| = 0.2-0.5 | Small | Weak but may help in combination with others |
| \|d\| < 0.2 | Negligible | Limited predictive value alone |

**🎯 Actionable Insights:**
- **Features with large effects** are your best predictors - ensure they're included in your model
- **Direction matters**: "Higher in retained" means customers with high values tend to stay; use this for threshold-based business rules
- **Features with small/negligible effects** may still be useful in combination or as interaction terms

**⚠️ Cautions:**
- Effect size assumes roughly normal distributions - check skewness in notebook 03
- Large effects could be due to confounding variables - validate with domain knowledge
- Correlation ≠ causation: high engagement may not *cause* retention

### Box Plot Visualization

**📈 How to Read the Box Plots Below:**
- **Well-separated boxes** (little/no overlap) → Feature clearly distinguishes retained vs churned
- **Different medians** (center lines at different heights) → Groups have different typical values
- **Many outliers in one group** → May indicate subpopulations worth investigating

In [ ]:
# @cr:code name='compute_feature_importance' id=f6986cb3
_box_data = None

if findings.target_column and findings.target_column in df.columns:
    target = findings.target_column

    feature_cols = [
        name for name, col in findings.columns.items()
        if col.inferred_type in [ColumnType.NUMERIC_CONTINUOUS, ColumnType.NUMERIC_DISCRETE]
        and name != target
        and name not in TEMPORAL_METADATA_COLS
    ]

    if feature_cols:
        n_features = min(len(feature_cols), MAX_BOX_FEATURES)
        plot_cols = feature_cols[:n_features]

        # Collect box-plot columns + target in a single call
        _box_data = df[plot_cols + [target]].to_numpy() if hasattr(df, 'to_spark') else df[plot_cols + [target]].values
        _box_target = _box_data[:, -1]

        fig = make_subplots(
            rows=1, cols=n_features,
            subplot_titles=plot_cols,
            horizontal_spacing=0.05
        )

        for i, col in enumerate(plot_cols):
            col_num = i + 1
            col_vals = _box_data[:, i]

            retained_mask = _box_target == 1
            retained_vals = col_vals[retained_mask]
            retained_data = retained_vals[~np.isnan(retained_vals)]

            churned_mask = _box_target == 0
            churned_vals = col_vals[churned_mask]
            churned_data = churned_vals[~np.isnan(churned_vals)]

            fig.add_trace(
                go.Box(
                    y=retained_data,
                    name='Retained',
                    fillcolor='rgba(46, 204, 113, 0.7)',
                    line=dict(color='#1e8449', width=2),
                    marker=dict(color='rgba(46, 204, 113, 0.5)', size=5,
                                line=dict(color='#1e8449', width=1)),
                    boxpoints='outliers', width=0.35,
                    showlegend=(i == 0), legendgroup='retained', offsetgroup='retained'
                ),
                row=1, col=col_num
            )

            fig.add_trace(
                go.Box(
                    y=churned_data,
                    name='Churned',
                    fillcolor='rgba(231, 76, 60, 0.7)',
                    line=dict(color='#922b21', width=2),
                    marker=dict(color='rgba(231, 76, 60, 0.5)', size=5,
                                line=dict(color='#922b21', width=1)),
                    boxpoints='outliers', width=0.35,
                    showlegend=(i == 0), legendgroup='churned', offsetgroup='churned'
                ),
                row=1, col=col_num
            )

        fig.update_layout(
            height=450,
            title_text="Feature Distributions: Retained (Green) vs Churned (Red)",
            template='plotly_white', showlegend=True,
            legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
            boxmode='group', boxgap=0.3, boxgroupgap=0.1
        )
        fig.update_xaxes(showticklabels=False)
        display_figure(fig)

        # Reuse bulk class_stats for mean comparison (no extra Spark jobs)
        print("\n📊 MEAN COMPARISON BY RETENTION STATUS:")
        print("-" * 70)
        for col in plot_cols:
            if _effect_sizes_result and col in _effect_sizes_result.class_stats:
                stats = _effect_sizes_result.class_stats[col]
                retained_mean = stats["mean_1"]
                churned_mean = stats["mean_0"]
            else:
                retained_mean = float(df[df[target] == 1][col].mean())
                churned_mean = float(df[df[target] == 0][col].mean())
            diff_pct = ((retained_mean - churned_mean) / churned_mean * 100) if churned_mean != 0 else 0
            print(f"  {col}:")
            print(f"     Retained: {retained_mean:.2f}  |  Churned: {churned_mean:.2f}  |  Diff: {diff_pct:+.1f}%")

[//]: # (cr:doc name='5_5_feature_target_correlations' id=763d38a5)
## 5.5 Feature-Target Correlations

Ranks all numeric features by their linear association with the target. Features with |r| > 0.3 are moderately predictive; those above 0.5 are strong candidates for the prioritised feature set. The ranking also feeds the interaction-detection step below — pairs of individually predictive features are tested for multiplicative effects that neither captures alone.

In [ ]:
# @cr:code name='analyze_interactions' id=81ec4d57
if findings.target_column and findings.target_column in df.columns:
    target = findings.target_column
    feature_cols = [
        name for name, col in findings.columns.items()
        if col.inferred_type in [ColumnType.NUMERIC_CONTINUOUS, ColumnType.NUMERIC_DISCRETE]
        and name != target
        and name not in TEMPORAL_METADATA_COLS
    ]

    if feature_cols:
        # Reuse target correlations from cell 9; compute only for any missing columns
        missing = [c for c in feature_cols if c not in _target_corrs]
        if missing:
            _target_corrs.update(bulk_corr_with_target(df, missing, target))
        target_corrs = {c: _target_corrs[c] for c in feature_cols if c in _target_corrs}

        correlations = [
            {"Feature": col, "Correlation": target_corrs.get(col, float('nan'))}
            for col in feature_cols
            if col in target_corrs
        ]

        corr_df = native_pd.DataFrame(correlations).sort_values("Correlation", key=abs, ascending=False)
        corr_df = corr_df.head(MAX_TARGET_CORR_BARS)

        fig = charts.bar_chart(
            corr_df["Feature"].tolist(),
            corr_df["Correlation"].tolist(),
            title=f"Feature Correlations with {target}"
        )
        display_figure(fig)
else:
    print("Target column not available for correlation analysis.")

[//]: # (cr:doc name='5_6_categorical_feature_analysis' id=220b8f56)
## 5.6 Categorical Feature Analysis

Measures the association between each categorical feature and the target using Cramer's V and per-category retention-rate lift. Categories with retention rates far from the population average identify high-risk segments that need adequate representation in training data. Features with high Cramer's V (>= 0.3) are strong predictors that should survive feature selection; those near zero add no signal.

| Metric | Purpose |
|--------|---------|
| **Retention Rate** | Identifies which categories are at highest risk |
| **Lift** (vs overall rate) | Quantifies how much a category deviates from average |
| **Cramer's V** | Overall strength of association (0–1, like correlation) |

In [ ]:
# @cr:code name='analyze_categorical_target' id=3e26270f
from customer_retention.stages.profiling import CategoricalTargetAnalyzer

_cat_results = {}

if findings.target_column:
    target = findings.target_column
    overall_retention = df[target].mean()

    categorical_cols = [c for c in cc.categorical
                        if findings.columns[c].inferred_type != ColumnType.CATEGORICAL_CYCLICAL]

    print("=" * 80)
    print("CATEGORICAL FEATURE ANALYSIS")
    print("=" * 80)
    print(f"Overall retention rate: {overall_retention:.1%}")

    if categorical_cols:
        cat_analyzer = CategoricalTargetAnalyzer(min_samples_per_category=10)

        # Single pass: analyze all columns, collect results for summary + details
        _cat_results = {col: cat_analyzer.analyze(df, col, target) for col in categorical_cols}

        summary_rows = []
        for col, result in _cat_results.items():
            summary_rows.append({
                'feature': col, 'n_categories': result.n_categories,
                'cramers_v': result.cramers_v, 'p_value': result.p_value,
                'effect_strength': result.effect_strength,
                'high_risk_count': len(result.high_risk_categories),
                'low_risk_count': len(result.low_risk_categories),
            })
        summary_df = native_pd.DataFrame(summary_rows).sort_values('cramers_v', ascending=False)

        print("\n\U0001f4c8 Categorical Feature Strength (Cram\u00e9r's V):")
        print("-" * 60)
        for _, row in summary_df.iterrows():
            if row["cramers_v"] >= 0.3:
                strength = "Strong"
                emoji = "\U0001f534"
            elif row["cramers_v"] >= 0.1:
                strength = "Moderate"
                emoji = "\U0001f7e1"
            else:
                strength = "Weak"
                emoji = "\U0001f7e2"
            sig = "***" if row["p_value"] < 0.001 else "**" if row["p_value"] < 0.01 else "*" if row["p_value"] < 0.05 else ""
            print(f"  {emoji} {row['feature']}: V={row['cramers_v']:.3f} ({strength}) {sig}")

        # Detailed analysis reusing pre-computed results
        for col_name in categorical_cols[:MAX_CATEGORICAL_DETAILS]:
            result = _cat_results[col_name]

            print(f"\n{'='*60}")
            print(f"\U0001f4ca {col_name.upper()}")
            print("="*60)

            if len(result.category_stats) > 0:
                # Collect to native pandas for safe formatting
                cat_stats = result.category_stats
                if hasattr(cat_stats, 'to_spark'):
                    cat_stats = cat_stats.to_pandas()

                display_stats = cat_stats[['category', 'total_count', 'retention_rate', 'lift', 'pct_of_total']].copy()
                display_stats['retention_rate'] = display_stats['retention_rate'].apply(lambda x: f"{x:.1%}")
                display_stats['lift'] = display_stats['lift'].apply(lambda x: f"{x:.2f}x")
                display_stats['pct_of_total'] = display_stats['pct_of_total'].apply(lambda x: f"{x:.1%}")
                display_stats.columns = [col_name, 'Count', 'Retention Rate', 'Lift', '% of Data']
                display_table(display_stats)

                categories = cat_stats['category'].tolist()
                retained_counts = cat_stats['retained_count'].tolist()
                churned_counts = cat_stats['churned_count'].tolist()

                fig = go.Figure()
                fig.add_trace(go.Bar(
                    name='Retained', x=categories, y=retained_counts,
                    marker_color='rgba(46, 204, 113, 0.8)',
                    text=[f"{r/(r+c)*100:.0f}%" for r, c in zip(retained_counts, churned_counts)],
                    textposition='inside', textfont=dict(color='white', size=12)
                ))
                fig.add_trace(go.Bar(
                    name='Churned', x=categories, y=churned_counts,
                    marker_color='rgba(231, 76, 60, 0.8)',
                    text=[f"{c/(r+c)*100:.0f}%" for r, c in zip(retained_counts, churned_counts)],
                    textposition='inside', textfont=dict(color='white', size=12)
                ))
                fig.update_layout(
                    barmode='stack', title=f"Retention by {col_name}",
                    xaxis_title=col_name, yaxis_title="Count",
                    template='plotly_white', height=350,
                    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
                )
                display_figure(fig)

                if result.high_risk_categories:
                    print("\n  \u26a0\ufe0f High-risk categories (lift < 0.9x):")
                    for cat in result.high_risk_categories:
                        cat_row = cat_stats[cat_stats['category'] == cat]
                        if len(cat_row) > 0:
                            cat_row = cat_row.iloc[0]
                            print(f"     \u2022 {cat}: {cat_row['retention_rate']:.1%} retention ({cat_row['lift']:.2f}x lift)")
    else:
        print("\n  \u2139\ufe0f No categorical columns detected.")
else:
    print("No target column available for categorical analysis.")

[//]: # (cr:doc name='5_7_scatter_plot_matrix_sample' id=003bfde3)
## 5.7 Scatter Plot Matrix (Sample)

Visual check for non-linear relationships that Pearson correlation misses. Curved trends suggest polynomial or log transforms; distinct clusters indicate natural customer segments that may benefit from segment-aware modelling. Only the top numeric features are plotted on a 1 000-row sample for performance.

In [ ]:
# @cr:code name='plot_scatter_pairs' id=528ee553
top_numeric = numeric_cols[:MAX_SCATTER_FEATURES] if len(numeric_cols) > MAX_SCATTER_FEATURES else numeric_cols

if len(top_numeric) >= 2:
    fig = charts.scatter_matrix(
        safe_sample(df[top_numeric], 1000),
        title="Scatter Plot Matrix (Sample)"
    )
    display_figure(fig)

[//]: # (cr:doc name='interpreting_the_scatter_matrix_above' id=d6311565)
### Interpreting the Scatter Matrix Above

**🎯 Key Questions to Answer:**

1. **Are any features redundant?**
   - Look for tight linear patterns → high correlation → consider dropping one
   - Cross-reference with high correlation pairs in section 4.3

2. **Are there natural customer segments?**
   - Distinct clusters suggest different customer types
   - Links to segment-aware outlier analysis in notebook 03

3. **Do relationships suggest feature engineering?**
   - Curved patterns → polynomial or interaction terms may help
   - Ratios between correlated features may be more predictive

4. **Are distributions suitable for linear models?**
   - Fan shapes or heavy skew → consider transformations
   - Outlier clusters → verify with segment analysis

**💡 Pro Tip:** Hover over points in the interactive plot to see exact values. Look for outliers that appear across multiple scatter plots - these may be influential observations worth investigating.

[//]: # (cr:doc name='5_8_datetime_feature_analysis' id=8145b868)
## 5.8 Datetime Feature Analysis

Temporal columns often carry strong retention signals — tenure, recency, and cohort effects. This step measures how retention varies over calendar periods (signup month, day-of-week) and continuous time features (days since last activity). Cohort effects identify periods where external factors (campaigns, economic shifts) altered customer behaviour, which the model must account for rather than treat as noise.

In [ ]:
# @cr:code name='analyze_temporal_target' id=a5eefead
from customer_retention.stages.profiling import TemporalTargetAnalyzer

datetime_cols = [
    name for name, col in findings.columns.items()
    if col.inferred_type == ColumnType.DATETIME
]

print("=" * 80)
print("DATETIME FEATURE ANALYSIS")
print("=" * 80)
print(f"Detected datetime columns: {datetime_cols}")

if datetime_cols and findings.target_column:
    target = findings.target_column
    overall_retention = float(df[target].mean())

    temporal_analyzer = TemporalTargetAnalyzer(min_samples_per_period=10)

    for col_name in datetime_cols[:MAX_DATETIME_DETAILS]:
        result = temporal_analyzer.analyze(df, col_name, target)

        print(f"\n{'='*60}")
        print(f"\U0001f4c5 {col_name.upper()}")
        print("="*60)

        if result.n_valid_dates == 0:
            print("  No valid dates found")
            continue

        print(f"  Date range: {result.min_date} to {result.max_date}")
        print(f"  Valid dates: {result.n_valid_dates:,}")

        # 1. Retention by Year (bounded extraction for Plotly)
        if len(result.yearly_stats) > 1:
            print(f"\n  \U0001f4ca Retention by Year: Trend is {result.yearly_trend}")

            periods = head_as_list(result.yearly_stats['period'].astype(str), 30)
            rates = head_as_list(result.yearly_stats['retention_rate'], 30)
            counts = head_as_list(result.yearly_stats['count'], 30)

            fig = make_subplots(rows=1, cols=2, subplot_titles=["Retention Rate by Year", "Customer Count by Year"],
                               column_widths=[0.6, 0.4])

            fig.add_trace(
                go.Scatter(
                    x=periods, y=rates,
                    mode='lines+markers', name='Retention Rate',
                    line=dict(color='#3498db', width=3), marker=dict(size=10)
                ),
                row=1, col=1
            )
            fig.add_hline(y=overall_retention, line_dash="dash", line_color="gray",
                         annotation_text=f"Overall: {overall_retention:.1%}", row=1, col=1)

            fig.add_trace(
                go.Bar(
                    x=periods, y=counts,
                    name='Count', marker_color='rgba(52, 152, 219, 0.6)'
                ),
                row=1, col=2
            )

            fig.update_layout(height=350, template='plotly_white', showlegend=False)
            fig.update_yaxes(tickformat='.0%', row=1, col=1)
            display_figure(fig)

        # 2. Retention by Month (bounded extraction for Plotly)
        if len(result.monthly_stats) > 1:
            print("\n  \U0001f4ca Retention by Month (Seasonality):")

            month_names = head_as_list(result.monthly_stats['month_name'], 12)
            month_rates = head_as_list(result.monthly_stats['retention_rate'], 12)
            colors = ['rgba(46, 204, 113, 0.7)' if r >= overall_retention else 'rgba(231, 76, 60, 0.7)'
                     for r in month_rates]

            fig = go.Figure()
            fig.add_trace(go.Bar(
                x=month_names, y=month_rates,
                marker_color=colors,
                text=[f"{r:.0%}" for r in month_rates],
                textposition='outside'
            ))
            fig.add_hline(y=overall_retention, line_dash="dash", line_color="gray",
                         annotation_text=f"Overall: {overall_retention:.1%}")

            fig.update_layout(
                title=f"Monthly Retention Pattern ({col_name})",
                xaxis_title="Month", yaxis_title="Retention Rate",
                template='plotly_white', height=350, yaxis_tickformat='.0%'
            )
            display_figure(fig)

            if result.seasonal_spread > 0.05:
                print(f"  \U0001f4c8 Seasonal spread: {result.seasonal_spread:.1%}")
                print(f"     Best month: {result.best_month}")
                print(f"     Worst month: {result.worst_month}")

        # 3. Retention by Day of Week (bounded extraction for Plotly)
        if len(result.dow_stats) > 1:
            print("\n  \U0001f4ca Retention by Day of Week:")

            day_names = head_as_list(result.dow_stats['day_name'], 7)
            dow_rates = head_as_list(result.dow_stats['retention_rate'], 7)
            colors = ['rgba(46, 204, 113, 0.7)' if r >= overall_retention else 'rgba(231, 76, 60, 0.7)'
                     for r in dow_rates]

            fig = go.Figure()
            fig.add_trace(go.Bar(
                x=day_names, y=dow_rates,
                marker_color=colors,
                text=[f"{r:.0%}" for r in dow_rates],
                textposition='outside'
            ))
            fig.add_hline(y=overall_retention, line_dash="dash", line_color="gray")

            fig.update_layout(
                title=f"Day of Week Pattern ({col_name})",
                xaxis_title="Day of Week", yaxis_title="Retention Rate",
                template='plotly_white', height=300, yaxis_tickformat='.0%'
            )
            display_figure(fig)
else:
    if not datetime_cols:
        print("\n  \u2139\ufe0f No datetime columns detected in this dataset.")
        print("     Consider adding date parsing in notebook 01 if dates exist as strings.")
    else:
        print("\n  \u2139\ufe0f No target column available for retention analysis.")


[//]: # (cr:doc name='5_9_actionable_recommendations_summary' id=8ff2c92e)
## 5.9 Actionable Recommendations

Consolidates all evidence from sections 5.2–5.8 into pipeline-actionable recommendations. Each recommendation is categorised and prioritised so that NB08 and NB10 can consume them programmatically — strong predictors are kept, redundant pairs are pruned, and interaction candidates are flagged for gold-layer feature engineering.

In [ ]:
# @cr:code name='generate_recommendations' id=ce9806ee
# Generate comprehensive actionable recommendations
recommender = RelationshipRecommender()

numeric_features = [c for c in cc.numeric if c != findings.target_column]
categorical_features = [c for c in cc.categorical
                        if findings.columns[c].inferred_type != ColumnType.CATEGORICAL_CYCLICAL]

# Run comprehensive analysis — reuse all precomputed results (no recomputation)
analysis_summary = recommender.analyze(
    df,
    numeric_cols=numeric_features,
    categorical_cols=categorical_features,
    target_col=findings.target_column,
    correlation_matrix=corr_matrix,
    effect_sizes=_effect_sizes_result.effect_sizes if _effect_sizes_result else None,
    categorical_results=_cat_results if categorical_features else None,
)

print("=" * 80)
print("ACTIONABLE RECOMMENDATIONS FROM RELATIONSHIP ANALYSIS")
print("=" * 80)

# Group recommendations by category
grouped_recs = analysis_summary.recommendations_by_category
high_priority = analysis_summary.high_priority_actions

if high_priority:
    print(f"\n🔴 HIGH PRIORITY ACTIONS ({len(high_priority)}):")
    print("-" * 60)
    for rec in high_priority:
        print(f"\n  📌 {rec.title}")
        print(f"     {rec.description}")
        print(f"     → Action: {rec.action}")
        if rec.affected_features:
            print(f"     → Features: {', '.join(rec.affected_features[:5])}")

# Persist recommendations to registry
for pair in analysis_summary.multicollinear_pairs:
    registry.add_gold_drop_multicollinear(
        column=pair["feature1"], correlated_with=pair["feature2"],
        correlation=pair["correlation"],
        rationale=f"High correlation ({pair['correlation']:.2f}) - consider dropping one",
        source_notebook="05_relationship_analysis"
    )

for predictor in analysis_summary.strong_predictors:
    registry.add_gold_prioritize_feature(
        column=predictor["feature"], effect_size=predictor["effect_size"],
        correlation=predictor["correlation"],
        rationale=f"Strong predictor with effect size {predictor['effect_size']:.2f}",
        source_notebook="05_relationship_analysis"
    )

for weak_col in analysis_summary.weak_predictors[:10]:
    registry.add_gold_drop_weak(
        column=weak_col, effect_size=0.0, correlation=0.0,
        rationale="Negligible predictive power",
        source_notebook="05_relationship_analysis"
    )

# Persist ratio feature recommendations
for rec in grouped_recs.get(RecommendationCategory.FEATURE_ENGINEERING, []):
    if "ratio" in rec.title.lower():
        for col1, col2, _corr in rec.evidence.get("moderate_pairs", []):
            registry.add_silver_ratio(
                column=f"{col1}_to_{col2}_ratio",
                numerator=col1, denominator=col2,
                rationale=rec.description, source_notebook="05_relationship_analysis"
            )
    elif "interaction" in rec.title.lower() and len(rec.affected_features) >= 2:
        for i, f1 in enumerate(rec.affected_features[:3]):
            for f2 in rec.affected_features[i+1:4]:
                registry.add_silver_interaction(
                    column=f"{f1}_x_{f2}", features=[f1, f2],
                    rationale=rec.description, source_notebook="05_relationship_analysis"
                )

# Store for findings metadata
findings.metadata["relationship_analysis"] = {
    "n_recommendations": len(analysis_summary.recommendations),
    "n_high_priority": len(high_priority),
    "strong_predictors": [p["feature"] for p in analysis_summary.strong_predictors],
    "weak_predictors": analysis_summary.weak_predictors[:5],
    "multicollinear_pairs": [(p["feature1"], p["feature2"]) for p in analysis_summary.multicollinear_pairs],
}

print(f"\n✅ Persisted {len(analysis_summary.multicollinear_pairs)} multicollinearity recommendations")
print(f"✅ Persisted {len(analysis_summary.strong_predictors)} strong predictor recommendations")
print(f"✅ Persisted {min(len(analysis_summary.weak_predictors), 10)} weak predictor recommendations")

[//]: # (cr:doc name='5_9_1_feature_selection_recommendations' id=19c6dcf4)
### 5.9.1 Feature Selection Recommendations

Classifies features into three tiers based on the evidence collected above: **strong predictors** (high effect size or target correlation) to prioritise, **weak predictors** (negligible effect size) to consider dropping, and **multicollinear pairs** where one member should be removed. These recommendations are saved to `merged/recommendations.yaml` and consumed by NB08's `APPLY_NB05_DROPS` gate.

In [ ]:
# @cr:code name='display_feature_selection' id=96df0793
# Feature Selection Recommendations
selection_recs = grouped_recs.get(RecommendationCategory.FEATURE_SELECTION, [])

print("=" * 70)
print("FEATURE SELECTION")
print("=" * 70)

# Strong predictors summary
if analysis_summary.strong_predictors:
    print("\n✅ STRONG PREDICTORS (prioritize these):")
    strong_df = native_pd.DataFrame(analysis_summary.strong_predictors)
    strong_df["effect_size"] = strong_df["effect_size"].apply(lambda x: f"{x:+.3f}")
    strong_df["correlation"] = strong_df["correlation"].apply(lambda x: f"{x:+.3f}")
    strong_df = strong_df.sort_values("effect_size", key=lambda x: x.str.replace("+", "").astype(float).abs(), ascending=False)
    display_table(strong_df)

    print("\n   💡 These features show strong discrimination between retained/churned customers.")
    print("   → Ensure they're included in your model")
    print("   → Check for data quality issues that could inflate their importance")

# Weak predictors summary
if analysis_summary.weak_predictors:
    print(f"\n⚪ WEAK PREDICTORS (consider dropping): {', '.join(analysis_summary.weak_predictors[:5])}")
    print("   → Low individual predictive power, but may help in combination")

# Multicollinearity summary
if analysis_summary.multicollinear_pairs:
    print("\n⚠️ MULTICOLLINEAR PAIRS (drop one from each pair for linear models):")
    for pair in analysis_summary.multicollinear_pairs:
        print(f"   • {pair['feature1']} ↔ {pair['feature2']}: r = {pair['correlation']:.2f}")
    print("\n   💡 For each pair, keep the feature with:")
    print("      - Stronger business meaning")
    print("      - Higher target correlation")
    print("      - Fewer missing values")

# Display all feature selection recommendations
if selection_recs:
    print("\n" + "-" * 70)
    print("DETAILED RECOMMENDATIONS:")
    for rec in selection_recs:
        priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "🟢"
        print(f"\n{priority_icon} {rec.title}")
        print(f"   {rec.description}")
        print(f"   → {rec.action}")

[//]: # (cr:doc name='5_9_2_stratification_recommendations' id=550ab7af)
### 5.9.2 Stratification Recommendations

Identifies categorical segments with extreme retention rates that need adequate representation in train and test sets. Without stratification, rare high-risk segments may end up entirely in one split, biasing evaluation metrics and hiding model weaknesses for those populations.

In [ ]:
# @cr:code name='display_stratification' id=529bd986
# Stratification Recommendations
strat_recs = grouped_recs.get(RecommendationCategory.STRATIFICATION, [])

print("=" * 70)
print("STRATIFICATION (Train/Test Split Strategy)")
print("=" * 70)

# High-risk segments
if analysis_summary.high_risk_segments:
    print("\n🎯 HIGH-RISK SEGMENTS (ensure representation in training data):")
    risk_df = native_pd.DataFrame(analysis_summary.high_risk_segments)
    risk_df["retention_rate"] = risk_df["retention_rate"].apply(lambda x: f"{x:.1%}")
    risk_df["lift"] = risk_df["lift"].apply(lambda x: f"{x:.2f}x")
    display_table(risk_df[["feature", "segment", "count", "retention_rate", "lift"]])

    print("\n   💡 These segments have below-average retention.")
    print("   → Ensure they're adequately represented in both train and test sets")
    print("   → Consider oversampling or class weights in modeling")

# Display all stratification recommendations
if strat_recs:
    print("\n" + "-" * 70)
    print("STRATIFICATION RECOMMENDATIONS:")
    for rec in strat_recs:
        priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "🟢"
        print(f"\n{priority_icon} {rec.title}")
        print(f"   {rec.description}")
        print(f"   → {rec.action}")
else:
    print("\n✅ No special stratification requirements detected.")
    print("   Standard stratified split by target variable is sufficient.")

[//]: # (cr:doc name='5_9_3_model_selection_recommendations' id=f602b254)
### 5.9.3 Model Selection Recommendations

Suggests model families based on the data characteristics observed: high multicollinearity favours tree-based models, strong linear relationships suit logistic regression, and class imbalance requires `class_weight='balanced'` or equivalent resampling. These guide the model grid in NB08.

In [ ]:
# @cr:code name='display_model_selection' id=5b7c3bff
# Model Selection Recommendations
model_recs = grouped_recs.get(RecommendationCategory.MODEL_SELECTION, [])

print("=" * 70)
print("MODEL SELECTION")
print("=" * 70)

if model_recs:
    for rec in model_recs:
        priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "🟢"
        print(f"\n{priority_icon} {rec.title}")
        print(f"   {rec.description}")
        print(f"   → {rec.action}")

# Summary recommendations based on data characteristics
print("\n" + "-" * 70)
print("RECOMMENDED MODELING APPROACH:")

has_multicollinearity = len(analysis_summary.multicollinear_pairs) > 0
has_strong_linear = len([p for p in analysis_summary.strong_predictors if abs(p.get("effect_size", 0)) >= 0.5]) > 0
has_categoricals = len(categorical_features) > 0

if has_strong_linear and not has_multicollinearity:
    print("\n✅ RECOMMENDED: Start with Logistic Regression")
    print("   • Strong linear relationships detected")
    print("   • Interpretable coefficients for business insights")
    print("   • Fast training and inference")
    print("   • Then compare with tree-based ensemble for potential improvement")
elif has_multicollinearity:
    print("\n✅ RECOMMENDED: Start with Random Forest or XGBoost")
    print("   • Multicollinearity present - tree models handle it naturally")
    print("   • Can keep all features without VIF analysis")
    print("   • Use feature importance to understand contributions")
else:
    print("\n✅ RECOMMENDED: Compare Linear and Tree-Based Models")
    print("   • No clear linear dominance - test both approaches")
    print("   • Logistic Regression for interpretability baseline")
    print("   • Random Forest/XGBoost for potential accuracy gain")

if has_categoricals:
    print("\n💡 CATEGORICAL HANDLING:")
    print("   • For tree models: Consider CatBoost or LightGBM with native categorical support")
    print("   • For linear models: Use target encoding for high-cardinality features")

[//]: # (cr:doc name='5_9_4_feature_engineering_recommendations' id=6de9d98c)
### 5.9.4 Feature Engineering Recommendations

Identifies opportunities for derived features based on the relationship patterns above: interaction terms for feature pairs that are individually predictive, ratio features for correlated pairs where the ratio may be more informative than either alone, and polynomial terms where scatter plots show non-linear trends. These feed the gold-layer transform steps in NB08 and the generated pipeline.

In [ ]:
# @cr:code name='display_feature_engineering' id=89f87163
# Feature Engineering Recommendations
eng_recs = grouped_recs.get(RecommendationCategory.FEATURE_ENGINEERING, [])

print("=" * 70)
print("FEATURE ENGINEERING")
print("=" * 70)

if eng_recs:
    for rec in eng_recs:
        priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "🟢"
        print(f"\n{priority_icon} {rec.title}")
        print(f"   {rec.description}")
        print(f"   → {rec.action}")
        if rec.affected_features:
            print(f"   → Features: {', '.join(rec.affected_features[:5])}")
else:
    print("\n✅ No specific feature engineering recommendations based on correlation patterns.")
    print("   Consider domain-specific features based on business knowledge.")

# Additional suggestions based on strong predictors
if analysis_summary.strong_predictors:
    print("\n" + "-" * 70)
    print("POTENTIAL INTERACTION FEATURES:")
    strong_features = [p["feature"] for p in analysis_summary.strong_predictors[:5]]
    if len(strong_features) >= 2:
        print("\n   Based on strong predictors, consider interactions between:")
        for i, f1 in enumerate(strong_features[:3]):
            for f2 in strong_features[i+1:4]:
                print(f"   • {f1} × {f2}")
        print("\n   💡 Tree-based models discover interactions automatically.")
        print("   → For linear models, create explicit interaction columns.")

[//]: # (cr:doc name='5_9_4b_statistical_feature_selection' id=33b5291b)
### 5.9.4b Statistical Feature Selection

Applies automated variance and correlation filters to the feature set. Near-constant features (variance below threshold) add no discriminative power and inflate dimensionality. Among highly correlated pairs (above the correlation threshold), the weaker predictor is dropped. Results are persisted as `drop_multicollinear` and `drop_weak` recommendations in `merged/recommendations.yaml` — NB08 reads these via its `APPLY_NB05_DROPS` flag.

In [ ]:
# @cr:config name='feature_selection_config' id=a55eb187
FEATURE_SELECTION_ENABLED = True       # Master toggle for statistical feature selection
VARIANCE_SELECTION_ENABLED = True      # Drop near-constant features
VARIANCE_THRESHOLD = 0.01             # Minimum variance to keep a feature
CORRELATION_SELECTION_ENABLED = True   # Drop one of each highly-correlated pair
CORRELATION_THRESHOLD = 0.95          # Max pairwise correlation allowed


In [ ]:
# @cr:code name='run_statistical_feature_selection' id=2e6e3e63
if findings.target_column and findings.target_column in df.columns and FEATURE_SELECTION_ENABLED:
    target = findings.target_column
    from customer_retention.stages.features.feature_selector import run_selection_pipeline

    _var_thresh = VARIANCE_THRESHOLD if VARIANCE_SELECTION_ENABLED else 0.0
    _corr_thresh = CORRELATION_THRESHOLD if CORRELATION_SELECTION_ENABLED else 1.0

    _pre_count = len([c for c in df.columns if c != target])
    _sel_result = run_selection_pipeline(
        df, target_column=target,
        variance_threshold=_var_thresh,
        correlation_threshold=_corr_thresh,
        l1_enabled=False,
        precomputed_corr_matrix=corr_matrix,
    )

    for _feat, _reason in _sel_result.drop_reasons.items():
        if "variance" in _reason.lower():
            registry.add_gold_drop_weak(
                _feat, 0.0, 0.0, _reason, "05_relationship_analysis"
            )
        elif "correlation" in _reason.lower():
            registry.add_gold_drop_multicollinear(
                _feat, "", 0.0, _reason, "05_relationship_analysis"
            )

    _all_analyzed = [c for c in df.columns if c != target]
    registry.set_feature_selection_config(
        variance_threshold=_var_thresh,
        correlation_threshold=_corr_thresh,
        analyzed_features=_all_analyzed,
    )

    print(f"\nStatistical feature selection: {_pre_count} -> {len(_sel_result.selected_features)} features")
    print(f"  Dropped: {len(_sel_result.dropped_features)}")
    _reason_counts = {}
    for _r in _sel_result.drop_reasons.values():
        _key = _r.split("(")[0].strip()
        _reason_counts[_key] = _reason_counts.get(_key, 0) + 1
    for _key, _cnt in _reason_counts.items():
        print(f"    {_key}: {_cnt}")

[//]: # (cr:doc name='5_9_5_recommendations_summary_table' id=baa8f741)
### 5.9.5 Manual Feature Exclusions and Summary

The `FEATURE_EXCLUSIONS` cell below lets you block specific aggregation functions for individual columns (e.g. exclude `mode` of a status column that leaks the target while keeping `count`). These per-dataset exclusions are written into the recommendations and enforced by the generated pipeline. The summary table shows all recommendations — automated and manual — in one view.

In [ ]:
# @cr:user_code name='feature_exclusions' id=bcd5a5bb
from customer_retention.generators.pipeline_generator.models import FeatureExclusion

# Per-dataset aggregation function blocking. Use blocked_funcs when some
# derivations of a column are safe but others leak the target.
#
# FEATURE_EXCLUSIONS = {
#     "my_dataset": [
#         FeatureExclusion(column="STATUS", blocked_funcs=["mode"], rationale="..."),
#         FeatureExclusion(column="SCORE", blocked_funcs=["mean", "sum"], rationale="..."),
#     ],
# }
FEATURE_EXCLUSIONS = {}

# Gold-level prefix exclusion. Columns kept at landing but whose datetime-
# derived aggregations ({COL}_*) and lag/velocity variants leak. Milestone
# features (days_to_milestone_*, *_progress_*) use a different naming
# pattern and are NOT affected.
#
# EXCLUDED_LEAKING_FEATURES = {
#     "my_dataset": ["END_DATE"],
# }
EXCLUDED_LEAKING_FEATURES = {}


In [ ]:
# @cr:code name='display_recommendations_summary' id=c1c8995a
# Create summary table of all recommendations
all_recs_data = []
for rec in analysis_summary.recommendations:
    all_recs_data.append({
        "Category": rec.category.value.replace("_", " ").title(),
        "Priority": rec.priority.upper(),
        "Recommendation": rec.title,
        "Action": rec.action[:80] + "..." if len(rec.action) > 80 else rec.action
    })

if all_recs_data:
    recs_df = native_pd.DataFrame(all_recs_data)

    # Sort by priority
    priority_order = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
    recs_df["_sort"] = recs_df["Priority"].map(priority_order)
    recs_df = recs_df.sort_values("_sort").drop("_sort", axis=1)

    print("=" * 80)
    print("ALL RECOMMENDATIONS SUMMARY")
    print("=" * 80)
    print(f"\nTotal: {len(recs_df)} recommendations")
    print(f"  \U0001f534 High priority: {len(recs_df[recs_df['Priority'] == 'HIGH'])}")
    print(f"  \U0001f7e1 Medium priority: {len(recs_df[recs_df['Priority'] == 'MEDIUM'])}")
    print(f"  \U0001f7e2 Low priority: {len(recs_df[recs_df['Priority'] == 'LOW'])}")

    display_table(recs_df)

# Save updated findings and recommendations registry
findings.save(FINDINGS_PATH)
registry.save(RECOMMENDATIONS_PATH)

print(f"\n\u2705 Findings updated with relationship analysis: {FINDINGS_PATH}")
print(f"\u2705 Recommendations registry saved: {RECOMMENDATIONS_PATH}")
print(f"   Total recommendations in registry: {len(registry.all_recommendations)}")

if _namespace:
    from customer_retention.analysis.auto_explorer.project_context import ProjectContext

    _namespace.merged_dir.mkdir(parents=True, exist_ok=True)
    _all_findings = _namespace.discover_all_findings(prefer_aggregated=True)
    _mgr = ExplorationManager(_namespace.merged_dir, findings_paths=_all_findings)

    _scaffold = []
    _ctx = None
    if _namespace.project_context_path.exists():
        _ctx = ProjectContext.load(_namespace.project_context_path)
        _scaffold = _ctx.merge_scaffold

    _multi = _mgr.create_multi_dataset_findings(merge_scaffold=_scaffold)
    if _ctx:
        for _name, _entry in _ctx.datasets.items():
            if _name in _multi.datasets:
                _multi.datasets[_name].raw_source_path = _entry.path

    if "FEATURE_EXCLUSIONS" in dir():
        for _ds_name, _excls in FEATURE_EXCLUSIONS.items():
            if _ds_name in _multi.datasets:
                _multi.datasets[_ds_name].feature_exclusions = _excls

    if "EXCLUDED_LEAKING_FEATURES" in dir():
        for _ds_name, _cols in EXCLUDED_LEAKING_FEATURES.items():
            if _ds_name in _multi.datasets:
                _multi.datasets[_ds_name].excluded_leaking_features = _cols

    _multi.save(str(_namespace.multi_dataset_findings_path))
    print(f"\n\u2705 Merged findings saved to {_namespace.merged_dir}")


In [ ]:
# @cr:code name='release_stage_memory' id=d16b656b
from customer_retention.core.compat import release_stage_memory

release_stage_memory()
del df, _box_data, _cat_results, corr_matrix, _effect_sizes_result, _target_corrs, analysis_summary

[//]: # (cr:doc name='summary_what_we_learned' id=ac445fe4)
---

## Summary: What We Learned

In this notebook, we analyzed feature relationships and generated **actionable recommendations** for modeling.

### Analysis Performed

**Numeric Features:**
1. **Correlation Matrix** - Identified multicollinearity issues between feature pairs
2. **Effect Sizes (Cohen's d)** - Quantified how well features discriminate retained vs churned
3. **Box Plots** - Visualized distribution differences between classes
4. **Feature-Target Correlations** - Ranked features by predictive power

**Categorical Features:**
5. **Cramér's V** - Measured association strength for categorical variables
6. **Retention by Category** - Identified high-risk segments
7. **Lift Analysis** - Found categories performing above/below average

**Datetime Features:**
8. **Cohort Analysis** - Retention trends by signup year
9. **Seasonality** - Monthly patterns in retention

### Actionable Recommendations Generated

| Category | What It Tells You | Impact on Pipeline |
|----------|-------------------|-------------------|
| **Feature Selection** | Which features to prioritize/drop | Reduces noise, improves interpretability |
| **Stratification** | How to split train/test | Ensures fair evaluation |
| **Model Selection** | Which algorithms to try first | Matches model to data |
| **Feature Engineering** | Interactions to create | Captures non-linear patterns |

### Key Metrics Reference

| Data Type | Effect Measure | Strong Signal |
|-----------|---------------|---------------|
| Numeric | Cohen's d | \|d\| ≥ 0.8 |
| Numeric | Correlation | \|r\| ≥ 0.5 |
| Categorical | Cramér's V | V ≥ 0.3 |
| Categorical | Lift | < 0.9x or > 1.1x |

---

## Recommended Actions Checklist

Based on the analysis above, here are the key actions to take:

- [ ] **Feature Selection**: Review strong/weak predictors and multicollinear pairs
- [ ] **Stratification**: Use stratified sampling with identified high-risk segments
- [ ] **Model Selection**: Start with recommended model type based on data characteristics
- [ ] **Feature Engineering**: Create interaction features between strong predictors

---

## Next Steps

Continue to **05_feature_opportunities.ipynb** to:
- Generate derived features (tenure, recency, engagement scores)
- Identify interaction features based on relationships found here
- Create business-relevant composite scores
- Review automated feature recommendations

[//]: # (cr:doc name='section' id=8d492aee)
> **Save Reminder:** Save this notebook (Ctrl+S / Cmd+S) before running the next one.
> The next notebook will automatically export this notebook's HTML documentation from the saved file.